In [1]:
import re
import numpy as np
import pandas as pd
import os


In [4]:
os.getcwd()

'/home/chaitu/Futurense/python/pandas_assignments'

In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../notebooks")
print(os.listdir(DATA_DIR))
members = pd.read_csv(
    DATA_DIR / "members.csv",
    parse_dates=["join_date"]
)

renewals = pd.read_csv(
    DATA_DIR / "renewal_opportunities.csv",
    parse_dates=["expiry_date", "renewal_deadline"]
)

events = pd.read_csv(
    DATA_DIR / "member_events.csv",
    parse_dates=["event_time"]
)

# Date columns supplied for the exercises.
members["join_year"] = members["join_date"].dt.year
renewals["expiry_year"] = renewals["expiry_date"].dt.year
renewals["expiry_month"] = renewals["expiry_date"].dt.month

['DATA_DICTIONARY.md', 'Kalpa_Plus_Student.ipynb', 'members.csv', 'renewal_opportunities.csv', 'member_events.csv']


1. Using members, group by acquisition_channel. Return:


- members, group by acquisition_channel
- average_annual_fee: mean annual fee
- earliest_join: earliest joining date.
- latest_join: latest joining date

In [4]:
members.groupby('acquisition_channel').agg(member_count=('member_id','count'),
                                           average_annual_fee=('annual_fee','mean'),
                                           earliest_join=('join_date','min'),
                                           latest_join=('join_date','max'))

,member_count,average_annual_fee,earliest_join,latest_join
acquisition_channel,,,,
APP,4328,1805.839187,2023-09-01,2026-10-01
STORE,1655,1799.302115,2023-09-01,2026-10-01
WEB,2597,1794.725838,2023-09-01,2026-10-01


In [13]:
members.groupby('acquisition_channel').agg()


,earliest_join,latest_join
acquisition_channel,,
APP,2023-09-01,2026-10-01
STORE,2023-09-01,2026-10-01
WEB,2023-09-01,2026-10-01


2. Using .loc, select members whose status_2026_10_09 is "PAID". Group them by city and return:

In [72]:
condition = members.loc[members['status_2026_10_09']=="PAID"]
condition.groupby('city').agg(
    paid_members=('member_id','count'),
average_annual_fee=('annual_fee','mean'),
    channels_present=('acquisition_channel','nunique'),
)

,paid_members,average_annual_fee,channels_present
city,,,
Ahmedabad,725,1799.689655,3
Bengaluru,771,1821.957198,3
Delhi,754,1786.798408,3
Gurugram,755,1782.443709,3
Hyderabad,789,1800.647655,3
Jaipur,778,1819.051414,3
Mumbai,754,1772.209549,3
Pune,698,1794.128940,3


3. Select members who were "PAID" on 9 October 2025 but "EXPIRED" on 9 October 2026. Group by acquisition_channel and return:

In [32]:
paid_and_expired = members.loc[(members['status_2025_10_09']=="PAID")&(members['status_2026_10_09']=="EXPIRED")]
paid_and_expired = paid_and_expired.groupby('acquisition_channel').agg(
    expired_members=('member_id', 'count'),
    average_annual_fee=('annual_fee','mean'),
)
paid_and_expired = paid_and_expired.loc[paid_and_expired['expired_members']>=300]
paid_and_expired

,expired_members,average_annual_fee
acquisition_channel,,
APP,619,1794.638126
WEB,391,1826.365729


4. Select members who joined in 2025 or 2026 and are "PAID" at the current snapshot. Group by both join_year and acquisition_channel. Return:

paid_members: number of members.
cities_present: number of distinct cities.
average_annual_fee: mean annual fee.

In [3]:
members['join_year'] = members['join_date'].dt.year
joined_and_paid = members.loc[
    ((members['join_year'].isin([2025,2026])))&(
    (members['status_2026_10_09']=="PAID"))
]
grouped = joined_and_paid.groupby(['acquisition_channel','join_year']).agg(
    paid_members= ('member_id','count'),
    cities_present=('city','nunique'),
    average_annual_fee=('annual_fee','mean'),
)
grouped


paid_members  cities_present  \
acquisition_channel join_year                                 
APP                 2025                681               8   
                    2026                924               8   
STORE               2025                238               8   
                    2026                351               8   
WEB                 2025                409               8   
                    2026                541               8   

                               average_annual_fee  
acquisition_channel join_year                      
APP                 2025              1798.559471  
                    2026              1807.441558  
STORE               2025              1822.529412  
                    2026              1795.296296  
WEB                 2025              1775.283619  
                    2026              1794.748614

Select currently paid members acquired through "APP" or "WEB". Group by city and return:

- paid_members: number of members.
- plans_present: number of distinct plans.
- average_annual_fee: mean annual fee.

Using .loc on the summary, display only cities with at least 500 paid members and an average annual fee above 1,750. Show only paid_members and average_annual_fee in the final output.


In [7]:
paid_members = members.loc[
    (members['acquisition_channel']=="APP")|(members['acquisition_channel']=="WEB")]
grouped_paid_members= paid_members.groupby('city').agg(
    paid_members=('member_id','count'),
    plans_present=('plan_code','nunique'),
    average_annual_fee=('annual_fee','mean'),
)

grouped_paid_members = grouped_paid_members.loc[
    (grouped_paid_members["paid_members"]>=500)&
    (grouped_paid_members["average_annual_fee"]>1750),
    ["paid_members","average_annual_fee"]

]
grouped_paid_members[['paid_members','average_annual_fee']]

,paid_members,average_annual_fee
city,,
Ahmedabad,817,1813.565483
Bengaluru,856,1822.598131
Delhi,872,1773.082569
Gurugram,843,1776.580071
Hyderabad,895,1818.553073
Jaipur,926,1830.533477
Mumbai,868,1782.410138
Pune,848,1793.811321


Using renewals, group by renewal_number and return:

- members_due: number of renewal records.
- average_fee: mean renewal fee.
- average_discount: mean offered discount, including zero discounts.
- maximum_discount: largest offered discount.

In [6]:
renewals.groupby('renewal_number').agg(
    members_due=('member_id','count'),
    average_fee= ('renewal_fee','mean'),
    average_discount=('offered_discount', 'mean'),
    maximum_discount=('offered_discount', 'max'),
 )

,members_due,average_fee,average_discount,maximum_discount
renewal_number,,,,
1,6380,1801.194357,14.490596,250
2,3400,1800.764706,19.205882,250
3,1800,1809.000000,21.083333,250


Select renewal records with expiry month September and expiry year 2025 or 2026. Group by expiry_year and return:

- members_due: number of renewal records.
- distinct_members: number of distinct member IDs.
- first_expiry: earliest expiry date.
- last_expiry: latest expiry date.

In [9]:
expiry_records = renewals.loc[(renewals['expiry_date'].dt.month==9)&
                              (renewals['expiry_date'].dt.year.isin([2025,2026]))]
expiry_records.groupby('expiry_year').agg(
    members_due=('member_id','count'),
    distinct_members=('member_id','nunique'),
    first_expiry=('expiry_date','min'),
    last_expiry=('expiry_date','max'),
)
# Check whether members_due and distinct_members agree in each year. What would a disagreement mean?



,members_due,distinct_members,first_expiry,last_expiry
expiry_year,,,,
2025,4000,4000,2025-09-01,2025-09-30
2026,4480,4480,2026-09-01,2026-09-30


In [10]:
expiry_records = renewals.loc[(renewals['expiry_date'].dt.month==9)&(renewals['expiry_date'].dt.year.isin([2025,2026]))&
                              (renewals['offered_discount']>0)]

expiry_records.groupby(['expiry_year','renewal_number']).agg(
    members_due=('member_id','count'),
    average_fee = ('renewal_fee','mean'),
    total_offered_discount = ('offered_discount', 'sum'),
)

members_due  average_fee  total_offered_discount
expiry_year renewal_number                                                  
2025        1                       160  1836.500000                   29400
            2                       228  1775.315789                   40500
2026        1                        94  1818.148936                   17100
            2                       140  1770.428571                   24800
            3                       211  1797.578199                   37950

Select September 2026 renewal records with offered_discount greater than zero. Group by renewal_number and return:

- discounted_offers: number of records.
- average_discount: mean offered discount.
- minimum_discount: smallest offered discount.
- maximum_discount: largest offered discount.

- Use .loc on the summary to keep only groups with at least 50 discounted offers.

In [8]:

discounts_received_customers = renewals.loc[(renewals['offered_discount']>0)&(renewals['expiry_date'].dt.year==2026)&(renewals['expiry_date'].dt.month==9)]
group = discounts_received_customers.groupby('renewal_number').agg(
    discounted_offers= ('renewal_number','count'),
    average_discount=('offered_discount','mean'),
    minimum_discount=('offered_discount','min'),
    maximum_discount=('offered_discount','max'),
)
group.loc[group['discounted_offers']>=50]

,discounted_offers,average_discount,minimum_discount,maximum_discount
renewal_number,,,,
1,94,181.914894,150,250
2,140,177.142857,150,250
3,211,179.857820,150,250


Select records whose expiry year is 2026 and whose renewal_deadline is before 10 October 2026. Group by renewal_number and return:

- members_due: number of records.
- earliest_deadline: earliest deadline.
- latest_deadline: latest deadline.\


Explain why this filter excludes renewal windows still open at the end of 9 October. Do not calculate renewal rates: this table alone does not contain successful payment outcomes.

In [14]:
filtered_records = renewals.loc[(renewals['expiry_date'].dt.year==2026)&
                                (renewals['renewal_deadline']<'10-10-2026')]
filtered_records.groupby('renewal_number').agg(
    members_due=('member_id','count'),
    earliest_deadline=('renewal_deadline','min'),
    latest_deadline=('renewal_deadline','max'),
)

,members_due,earliest_deadline,latest_deadline
renewal_number,,,
1,1280,2026-09-08,2026-10-07
2,1400,2026-09-08,2026-10-07
3,1800,2026-09-08,2026-10-07


Using events, group by event_type and return:

- event_count: number of events.
- distinct_members: number of distinct members.
- distinct_renewals: number of distinct renewal IDs.
- total_event_cost: sum of recorded event costs.



 Why can event_count exceed distinct_members?

In [15]:
events.groupby('event_type').agg(
    event_count=('event_id','count'),
    distinct_members=('member_id','nunique'),
    distinct_renewals=('renewal_id','nunique'),
    total_event_cost = ('event_cost','sum')
)

,event_count,distinct_members,distinct_renewals,total_event_cost
event_type,,,,
BENEFIT_USED,125995,6374,11494,4526444
PAYMENT_ATTEMPT,13308,5004,9820,116856
REMINDER_SENT,11580,6380,11580,11580
SUPPORT_CONTACT,1174,1174,1174,52830


Select only "PAYMENT_ATTEMPT" events with outcome "SUCCESS". Group by channel and return:

- successful_payments: number of successful payment events.
- cash_collected: sum of payment amounts.
- average_payment: mean payment amount.
- total_payment_event_cost: sum of recorded costs for these successful events.


 The cost total here excludes failed payment attempts and other event types.

In [17]:
filtered_events = events.loc[
    (events['event_type']=="PAYMENT_ATTEMPT")&
    (events['outcome']=="SUCCESS")

]
filtered_events.groupby('channel').agg(
    successful_payments= ("outcome","count"),
    cash_collected=("amount","sum"),
    average_payment=("amount","mean"),
    total_payment_event_cost=('event_cost','sum')
)

,successful_payments,cash_collected,average_payment,total_payment_event_cost
channel,,,,
APP,4522,8057578,1781.861566,54264
WEB,4502,7989698,1774.699689,54024


Select failed payment-attempt events. Group by both channel and detail_code. Return:

- failed_attempts: number of failed events.
- affected_members: number of distinct members.
- affected_renewals: number of distinct renewal IDs.
- total_failure_cost: sum of recorded costs for these failed events.


Using .loc on the aggregated result, keep only groups with at least 100 failed attempts and at least 40 affected members.

In [9]:
filtered_events = events.loc[
    (events['event_type']=="PAYMENT_ATTEMPT")&
    (events['outcome']=="FAILED")

]

filtered_events = filtered_events.groupby(['channel','detail_code']).agg(
    failed_attempts=("event_id",'count'),
    affected_members=('member_id','nunique'),
    affected_renewals=('renewal_id','nunique'),
    total_failure_cost=('event_cost','sum'),
)
filtered_events.loc[(filtered_events['failed_attempts']>=100)&(filtered_events['affected_members']>=40)]

failed_attempts  affected_members  \
channel detail_code                                                
APP     AUTHENTICATION_FAILED              704               636   
        INSUFFICIENT_FUNDS                 502               482   
        PSP_TIMEOUT                       1717              1048   
WEB     AUTHENTICATION_FAILED              401               391   
        INSUFFICIENT_FUNDS                 367               361   
        PSP_TIMEOUT                        593               558   

                               affected_renewals  total_failure_cost  
channel detail_code                                                   
APP     AUTHENTICATION_FAILED                645                1408  
        INSUFFICIENT_FUNDS                   492                1004  
        PSP_TIMEOUT                         1067                3434  
WEB     AUTHENTICATION_FAILED                400                 802  
        INSUFFICIENT_FUNDS                   366                 734  
        PSP_TIMEOUT                          569                1186

14
Select "BENEFIT_USED" events recorded through "APP" or "WEB". Group by detail_code and return:

uses: number of benefit-use events.
members_using: number of distinct members.
total_cost: sum of recorded event costs.
average_cost_per_use: mean event cost.
Using .loc on the summary, display only benefits whose average cost per use exceeds 30. Show only uses, members_using and average_cost_per_use

In [19]:
filtered_events = events.loc[
    (events['event_type']=="BENEFIT_USED")&
    (events['channel'].isin(["APP","WEB"]))

]

final_events = filtered_events.groupby('detail_code').agg(
    uses=('event_id','count'),
    members_using=('member_id','nunique'),
    total_cost=('event_cost','sum'),
    average_cost_per_use=('event_cost','mean'),
)
final_events.loc[final_events['average_cost_per_use']>30, ('uses','members_using','average_cost_per_use')]

,uses,members_using,average_cost_per_use
detail_code,,,
FREE_DELIVERY,81803,6360,45.057137


15
Select "SUPPORT_CONTACT" events whose detail code is either "FREE_DELIVERY_UNAVAILABLE" or "RENEWAL_HELP". Group by detail_code and return:

contacts: number of support contacts.
members_contacting: number of distinct members.
renewals_involved: number of distinct renewal IDs.
total_support_cost: sum of recorded support costs.
Explain why the number of members contacting support cannot automatically be treated as the number of lost members.

In [75]:
filtered_events = events.loc[
    (events['event_type']=="SUPPORT_CONTACT")&
    (events['detail_code'].isin(["FREE_DELIVERY_UNAVAILABLE","RENEWAL_HELP"]))

]

filtered_events.groupby('detail_code').agg(
    contacts=('event_id','count'),
    members_contacting=('member_id','nunique'),
    renewals_involved=('renewal_id','nunique'),
    total_support_cost=('event_cost','sum'),
)

,contacts,members_contacting,renewals_involved,total_support_cost
detail_code,,,,
FREE_DELIVERY_UNAVAILABLE,620,620,620,27900
RENEWAL_HELP,104,104,104,4680


In [24]:
import pandas as pd

df = pd.DataFrame({
    "City": ["Delhi", "Delhi", "Jaipur", "Jaipur", "Mumbai", "Mumbai"],
    "Product": ["Pen", "Notebook", "Pen", "Notebook", "Notebook", "Pen"],
    "Sales": [200, 500, 150, 350, 600, 400]
})

df


,City,Product,Sales
0,Delhi,Pen,200
1,Delhi,Notebook,500
2,Jaipur,Pen,150
3,Jaipur,Notebook,350
4,Mumbai,Notebook,600
5,Mumbai,Pen,400


In [22]:
# retry_rate: percentage of those renewal IDs with two or more payment attempts within that channel.
payments = events.loc[events['event_type']=="PAYMENT_ATTEMPT"]
payments.groupby('channel').agg(payment_attempts=('event_id','count'),
                                renewal_attempts=('renewal_id','nunique'),)

,payment_attempts,renewal_attempts
channel,,
APP,7445,5822
WEB,5863,5296


In [23]:
def count_renewal_id(ds):
  return (ds.value_counts()>1).sum()

payments.groupby('channel').agg(count_renewal_ids = ('renewal_id', count_renewal_id))

,count_renewal_ids
channel,
APP,1183
WEB,564


In [26]:
# For each city find out range of sales
def range_sales(ds):
    return  ds.max()-ds.min()

df.groupby('City').agg(range_sales=('Sales', range_sales))


,range_sales
City,
Delhi,300
Jaipur,200
Mumbai,200


In [27]:
# Using member_events.csv, group by event_type and find the difference between the highest and lowest event_cost in each group.
# Write a custom function named cost_range that receives a Series and returns:
# Maximum cost − minimum cost
# Use .groupby().agg() to return:
# - event_count: number of events.
# - cost_range: calculated using your custom function.

def range_event_cost(ds):
    return ds.max()-ds.min()

events.groupby('event_type').agg(event_count=('event_id','count'),
                                 cost_range=('event_cost',range_event_cost))






,event_count,cost_range
event_type,,
BENEFIT_USED,125995,57
PAYMENT_ATTEMPT,13308,10
REMINDER_SENT,11580,0
SUPPORT_CONTACT,1174,0


In [ ]:
# Using renewal_opportunities.csv, group by renewal_number and find the percentage of members offered a discount.
# Write a custom function named discount_percentage that receives the offered_discount Series and returns:
# Number of values greater than zero ÷ total number of values × 100

In [30]:
def discount_percentage(ds):
    return (ds>0).sum()/ ds.shape[0] * 100


renewals.groupby('renewal_number').agg(offered_discounts=('offered_discount',discount_percentage),)

,offered_discounts
renewal_number,
1,7.884013
2,10.823529
3,11.722222


In [ ]:
# Using member_events.csv, select only "PAYMENT_ATTEMPT" events. Group by channel and calculate the percentage of failed attempts.
# Write a custom function named failure_percentage that receives the outcome Series and returns:
# Number of "FAILED" values ÷ total number of values × 100
# Use groupby().agg() to return:
# - total_attempts: number of payment attempts.
# - failure_percentage: calculated using your function, rounded to two decimal places.

In [41]:
def failed_attempts(ds):
    return round((ds=='FAILED').sum()/len(ds) * 100,2)

payments = events.loc[events['event_type']=="PAYMENT_ATTEMPT"]


payments.groupby('channel').agg(
    total_attempts=('event_id','count'),
    failure_percentage = ('outcome',failed_attempts),
)


,total_attempts,failure_percentage
channel,,
APP,7445,39.26
WEB,5863,23.21


In [33]:
events.loc["event_type"]

event_id                 EV0023859
member_id                 MP000456
renewal_id                RN001802
event_time     2024-07-03 08:50:00
event_type            BENEFIT_USED
channel                        APP
outcome                    SUCCESS
detail_code           MEMBER_OFFER
amount                           0
event_cost                      10
Name: 0, dtype: object

In [ ]:
# Using member_events.csv, group by event_type and find the average cost of events whose event_cost is greater than ₹30.
# Write a custom function named average_high_cost that receives the event_cost Series and:
# - Selects costs greater than ₹30.
# - Returns their mean.
# - Returns 0 if no costs qualify.
# Use groupby().agg() to return:
# - total_events: number of events.
# - average_high_cost: calculated using your function.

In [51]:
# greater_than_30 = events.loc[events['event_cost']>30]
def average_high_cost(ds):
    high_cost = round((ds[ds>30]).mean(),2)
    if high_cost:
    #     return 0
    # else:
        return high_cost

    return 0


events.groupby('event_type').agg(
    total_events= ('event_id','count'),
    average_high_cost=('event_cost',average_high_cost),
)




,total_events,average_high_cost
event_type,,
BENEFIT_USED,125995,48.04
PAYMENT_ATTEMPT,13308,NaN
REMINDER_SENT,11580,NaN
SUPPORT_CONTACT,1174,45.00


In [52]:
df = pd.DataFrame({
    "City": ["Delhi", "Delhi", "Delhi", "Jaipur", "Jaipur", "Jaipur"],
    "Sales": [1000, 2000, 500, 1500, 800, 1200],
    "Cost": [700, 1400, 600, 1000, 900, 800]
})

In [53]:
def get_profitable_sales_mean(dff):
  return dff.loc[dff['Sales']>dff['Cost'], 'Sales'].mean()

In [54]:
df.groupby('City').apply(get_profitable_sales_mean)

City
Delhi     1500.0
Jaipur    1350.0
dtype: float64

In [55]:
df.groupby('City').apply(get_profitable_sales_mean).reset_index(
    name='profitable_avg_city')


,City,profitable_avg_city
0,Delhi,1500.0
1,Jaipur,1350.0


In [56]:
df = pd.DataFrame({
    "Department": ["Sales", "Sales", "Sales", "Support", "Support", "Support"],
    "Salary": [40000, 55000, 65000, 30000, 45000, 50000],
    "Experience": [1, 3, 5, 2, 4, 1]
})
# For each dept find avg salary of employees with atleast 3 years exp

In [62]:
def get_avg_salary(dff):
    return dff.loc[dff["Experience"]>=3, 'Salary'].mean()

In [63]:
df.groupby('Department').apply(get_avg_salary).reset_index(name='avg_salary')

,Department,avg_salary
0,Sales,60000.0
1,Support,45000.0


In [64]:

# Using member_events.csv, select only "PAYMENT_ATTEMPT" events.
# For each channel, calculate the average payment amount of successful attempts only.

# Write a function named successful_payment_mean(group) that:

# Receives the whole group DataFrame.
# Selects rows where outcome == "SUCCESS".
# Returns the mean of amount for those rows.
# Use groupby().apply() and name the result column average_successful_payment when resetting the index.
# Failed attempts must not contribute to the average.

In [71]:
def successful_payment_mean(x):
    # if (events['event_type']=="PAYMENT_ATTEMPT") and (events['outcome']=="SUCCESS"):
        return x.loc[x['outcome']=="SUCCESS",'amount'].mean()
#
success_payments = events.loc[
    (events['event_type']=="PAYMENT_ATTEMPT")

]



success_payments.groupby('channel').apply(successful_payment_mean)

channel
APP    1781.861566
WEB    1774.699689
dtype: float64

In [76]:
# Finance wants to know which payment channel costs Kalpa more for each successful renewal.
# Using member_events.csv, calculate the payment-processing cost per successful renewal for each channel. Include the costs of failed attempts as well—they still cost Kalpa money.
# Which channel appears more expensive, and why might comparing only its average cost per attempt be misleading?

In [92]:
payments = events.loc[events['event_type']=="PAYMENT_ATTEMPT"]

def get_cost_per_success(dff):
    total_cost= dff['event_cost'].sum()
    success = (dff['outcome']=="SUCCESS").sum()
    if success:
        return round(total_cost/success,2)
    return np.nan

# all_costs = payments['event_cost'].s
# print(all_costs)
# print("********")
# success_processing_cost = payments.loc[(events['outcome']=="SUCCESS"),'event_cost'].count()
# print(success_processing_cost)

payments.groupby('channel').apply(get_cost_per_success).reset_index(name='cost_per_success')
# success_processing_cost.head()


,channel,cost_per_success
0,APP,13.29
1,WEB,12.60


In [111]:
import pandas as pd

df = pd.DataFrame({
    "City": ["Delhi", "Jaipur", "Delhi", "Jaipur"],
    "Sales": [100, 200, 300, 600]
})

df

,City,Sales
0,Delhi,100
1,Jaipur,200
2,Delhi,300
3,Jaipur,600


In [94]:
# Contribution of each line in overall sales

In [95]:
total_sales = df["Sales"].sum()

In [104]:

df["Contribution_to_sales"] = df["Sales"]/total_sales
df


,City,Sales,Contribution_to_sales
0,Delhi,100,0.083333
1,Jaipur,200,0.166667
2,Delhi,300,0.250000
3,Jaipur,600,0.500000


In [100]:
# Contribution of each line in their city

In [105]:
dff = df[df["City"]=="Delhi"]
dff

,City,Sales,Contribution_to_sales
0,Delhi,100,0.083333
2,Delhi,300,0.250000


In [103]:
dff['Sales']/dff['Sales'].sum()

0    0.25
2    0.75
Name: Sales, dtype: float64

In [114]:
def get_sales_per_city(dff):
    return dff['Sales']/ dff["Sales"].sum()
df.groupby('City').apply(get_sales_per_city).reset_index().drop(columns='level_1')


,City,Sales
0,Delhi,0.25
1,Delhi,0.75
2,Jaipur,0.25
3,Jaipur,0.75


In [115]:
# 1. Identify unusually expensive events
# Within each event_type, calculate its average event_cost.
# Return a Boolean Series indicating whether each event costs more than its group’s average. Assign it to:
# events["above_average_cost"]

In [116]:
dff = df[df["City"] == "Delhi"]
dff

,City,Sales
0,Delhi,100
2,Delhi,300


In [121]:
dff['Sales'].agg('mean')

np.float64(200.0)

In [127]:
dff = events[events['event_type']=="PAYMENT_ATTEMPT"]
dff["event_cost"]>dff["event_cost"].agg('mean')
# dff.head()

19460     False
19933      True
19934      True
19935      True
19937      True
          ...  
152038    False
152039    False
152040    False
152041    False
152042    False
Name: event_cost, Length: 13308, dtype: bool

In [129]:
def add_avg_cost(dff):
    return dff["event_cost"]>dff["event_cost"].agg('mean')

events["flagged_events"] = events.groupby('event_type').apply(add_avg_cost).reset_index(drop=True)
events.head()

,event_id,member_id,renewal_id,event_time,event_type,channel,outcome,detail_code,amount,event_cost,flagged_events
0,EV0023859,MP000456,RN001802,2024-07-03 08:50:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,10,False
1,EV0005255,MP000612,RN000390,2024-07-03 11:34:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,27,False
2,EV0000747,MP001531,RN000060,2024-07-03 13:20:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,55,True
3,EV0029706,MP002497,RN002203,2024-07-03 17:39:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,53,True
4,EV0031926,MP001821,RN002456,2024-07-03 17:47:00,BENEFIT_USED,APP,SUCCESS,PRIORITY_SUPPORT,0,13,False


In [134]:
# 2. Find each event’s share of channel cost
# Within each channel, calculate what percentage of the channel’s total event cost each row contributes.
# Return a Series and assign it to:
# events["channel_cost_share_pct"]
dff = events[events['channel']=="APP"]
dff

,event_id,member_id,renewal_id,event_time,event_type,channel,outcome,detail_code,amount,event_cost,flagged_events
0,EV0023859,MP000456,RN001802,2024-07-03 08:50:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,10,False
1,EV0005255,MP000612,RN000390,2024-07-03 11:34:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,27,False
3,EV0029706,MP002497,RN002203,2024-07-03 17:39:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,53,True
4,EV0031926,MP001821,RN002456,2024-07-03 17:47:00,BENEFIT_USED,APP,SUCCESS,PRIORITY_SUPPORT,0,13,False
9,EV0003421,MP001411,RN000258,2024-07-03 20:37:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,48,True
...,...,...,...,...,...,...,...,...,...,...,...
152038,EV0149577,MP006146,RN011322,2026-10-09 11:00:00,PAYMENT_ATTEMPT,APP,FAILED,AUTHENTICATION_FAILED,1499,2,False
152039,EV0149615,MP005796,RN011324,2026-10-09 11:00:00,PAYMENT_ATTEMPT,APP,FAILED,INSUFFICIENT_FUNDS,2499,2,False
152040,EV0149636,MP006000,RN011325,2026-10-09 11:00:00,PAYMENT_ATTEMPT,APP,FAILED,PSP_TIMEOUT,2499,2,False
152041,EV0149759,MP006357,RN011331,2026-10-09 11:00:00,PAYMENT_ATTEMPT,APP,FAILED,PSP_TIMEOUT,1499,2,False


In [137]:
# dff = events['channel']=="APP"
# total_events = events


dff["channel_cost_share_pct"] = dff["event_cost"]/dff["event_cost"].sum()
dff

,event_id,member_id,renewal_id,event_time,event_type,channel,outcome,detail_code,amount,event_cost,flagged_events,channel_cost_share_pct
0,EV0023859,MP000456,RN001802,2024-07-03 08:50:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,10,False,4.293616e-06
1,EV0005255,MP000612,RN000390,2024-07-03 11:34:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,27,False,1.159276e-05
3,EV0029706,MP002497,RN002203,2024-07-03 17:39:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,53,True,2.275617e-05
4,EV0031926,MP001821,RN002456,2024-07-03 17:47:00,BENEFIT_USED,APP,SUCCESS,PRIORITY_SUPPORT,0,13,False,5.581701e-06
9,EV0003421,MP001411,RN000258,2024-07-03 20:37:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,48,True,2.060936e-05
...,...,...,...,...,...,...,...,...,...,...,...,...
152038,EV0149577,MP006146,RN011322,2026-10-09 11:00:00,PAYMENT_ATTEMPT,APP,FAILED,AUTHENTICATION_FAILED,1499,2,False,8.587233e-07
152039,EV0149615,MP005796,RN011324,2026-10-09 11:00:00,PAYMENT_ATTEMPT,APP,FAILED,INSUFFICIENT_FUNDS,2499,2,False,8.587233e-07
152040,EV0149636,MP006000,RN011325,2026-10-09 11:00:00,PAYMENT_ATTEMPT,APP,FAILED,PSP_TIMEOUT,2499,2,False,8.587233e-07
152041,EV0149759,MP006357,RN011331,2026-10-09 11:00:00,PAYMENT_ATTEMPT,APP,FAILED,PSP_TIMEOUT,1499,2,False,8.587233e-07


In [144]:
def channel_cost_share_pct(dff):
    return dff["event_cost"]/dff["event_cost"].sum()*100

In [145]:
events.groupby('channel').apply(channel_cost_share_pct)

channel        
APP      0         0.000429
         1         0.001159
         3         0.002276
         4         0.000558
         9         0.002061
                     ...   
WEB      152012    0.000519
         152021    0.000086
         152023    0.000086
         152031    0.000086
         152032    0.000086
Name: event_cost, Length: 152057, dtype: float64